# Rock-pick keypoint auto-labeling with CoTracker3

Prepare labels for downstream YOLO Pose training by clicking five points once per clip.
This notebook runs a pretrained tracker; it does **not** train a YOLO model.

1. Select **Runtime → Change runtime type → T4 GPU**.
2. Run **1–2** once. In Cell 1, set `PROJECT_DIR` to your Google Drive folder.
3. For each clip, run **3 → 4 → 5 → 6 → 7**: select, click, track, review, export.
4. Each ZIP contains **only the current clip** and is saved to `ANNOTATED_DIR` on Drive.

The five keypoints must be visible in the click frame. Use a `CLICK_FRAME` divisible by
`TRACK_STRIDE` (default 8). Inspect tracking drift before export. Cell 6 moves rejected
image/label pairs into a local `rejected/` folder; it does not alter source videos.

Local `/content` files disappear when Colab ends. Export every clip before closing.
Exports are annotation batches: create disjoint splits by source recording/session
before training. See the repository README and `docs/workflow.md`.


In [ ]:
# 1. Install notebook dependencies, configure Drive, and load the tracker.
# Keep Colab's preinstalled CUDA-compatible torch; do not replace it with a CPU build.
%pip -q install numpy opencv-python matplotlib ipywidgets einops timm

import os, json, base64, shutil, math, gc
from pathlib import Path
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Javascript
from google.colab.output import eval_js
from google.colab import drive

drive.mount("/content/drive")

# EDIT this path to match your Drive project. Videos remain read-only inputs.
PROJECT_DIR = Path("/content/drive/MyDrive/rockpick")
VIDEO_DIR = PROJECT_DIR / "Videos"
ANNOTATED_DIR = PROJECT_DIR / "Annotated"
assert PROJECT_DIR.is_dir(), f"Project folder not found: {PROJECT_DIR}"
assert VIDEO_DIR.is_dir(), f"Video folder not found: {VIDEO_DIR}"
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_EXTENSIONS = {".mov", ".mp4", ".m4v"}

assert torch.cuda.is_available(), "Select a GPU runtime, then rerun this cell."
device = "cuda"
# Pin the tracker code for this organized copy. Original notebook used floating main.
COTRACKER_REVISION = "82e02e8029753ad4ef13cf06be7f4fc5facdda4d"
model = torch.hub.load(
    f"facebookresearch/co-tracker:{COTRACKER_REVISION}",
    "cotracker3_offline",
).to(device).eval()
print("Input directory:", VIDEO_DIR)
print("ZIP output directory:", ANNOTATED_DIR)
print("CoTracker3 loaded on", torch.cuda.get_device_name(0))
print("Runtime versions:", {"torch": torch.__version__, "numpy": np.__version__, "opencv": cv2.__version__})


In [ ]:
# 2. Settings + helper functions (run once)

# Preserve this order in every annotation/export and downstream pose model.
KEYPOINTS = ["blunt-end", "metal-intersection", "sharp-end", "metal-handle-intersection", "handle"]
CLASS_ID, CLASS_NAME = 0, "rock_pick"

EVERY = 8          # save every 8th frame
PAD = 0.08         # bounding-box padding around the keypoints
MIN_VISIBLE = 3    # skip if fewer than this many points are visible
MAX_SIDE = 640     # downscale for tracking only
TRACK_STRIDE = 8   # track every 8th frame to fit GPU memory
OUTPUT_WIDTH = 640
OUTPUT_HEIGHT = 480
OUT = "out_640x480"  # keep fixed-size crops separate from previous output

# Automatic black-border detection. The crop is computed once per video and cached.
AUTO_CROP_SAMPLES = 8
BLACK_THRESHOLD = 18       # pixels <= this value count as black
EDGE_ACTIVE_RATIO = 0.02   # content must occupy at least 2% of an edge row/column
MIN_BORDER = 12            # ignore tiny detected borders
MIN_CONTENT_FRACTION = 0.50
CROP_PAD = 2               # retain two pixels around detected content
CROP_CACHE = {}

COLORS_BGR = [(0,140,255), (255,0,255), (255,229,0), (0,255,0), (48,48,255)]


def detect_content_box(video: str) -> tuple[int, int, int, int]:
    """Detect a stable non-black rectangle from several frames."""
    cap = cv2.VideoCapture(video)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if total <= 0 or width <= 0 or height <= 0:
        cap.release()
        raise RuntimeError(f"Invalid video metadata: {video}")

    sample_count = min(AUTO_CROP_SAMPLES, total)
    sample_indices = np.linspace(0, total - 1, sample_count, dtype=int)
    max_gray = np.zeros((height, width), dtype=np.uint8)
    valid_samples = 0

    for frame_idx in sample_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok or frame.shape[:2] != (height, width):
            continue
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        np.maximum(max_gray, gray, out=max_gray)
        valid_samples += 1
    cap.release()

    full_box = (0, 0, width, height)
    if valid_samples == 0:
        print("Auto crop: no readable sample frames; keeping the full frame.")
        return full_box

    active = max_gray > BLACK_THRESHOLD
    active_cols = np.flatnonzero(active.mean(axis=0) >= EDGE_ACTIVE_RATIO)
    active_rows = np.flatnonzero(active.mean(axis=1) >= EDGE_ACTIVE_RATIO)
    if len(active_cols) == 0 or len(active_rows) == 0:
        print("Auto crop: no reliable content region; keeping the full frame.")
        return full_box

    x0, x1 = int(active_cols[0]), int(active_cols[-1] + 1)
    y0, y1 = int(active_rows[0]), int(active_rows[-1] + 1)

    # Add a small safety pad and keep even dimensions/coordinates.
    x0 = max(0, ((x0 - CROP_PAD) // 2) * 2)
    y0 = max(0, ((y0 - CROP_PAD) // 2) * 2)
    x1 = min(width, ((x1 + CROP_PAD + 1) // 2) * 2)
    y1 = min(height, ((y1 + CROP_PAD + 1) // 2) * 2)

    # Do not crop tiny edge variations.
    if x0 < MIN_BORDER:
        x0 = 0
    if y0 < MIN_BORDER:
        y0 = 0
    if width - x1 < MIN_BORDER:
        x1 = width
    if height - y1 < MIN_BORDER:
        y1 = height

    crop_width, crop_height = x1 - x0, y1 - y0
    if (crop_width < width * MIN_CONTENT_FRACTION or
            crop_height < height * MIN_CONTENT_FRACTION):
        print("Auto crop: detected region was implausibly small; keeping the full frame.")
        return full_box

    box = (x0, y0, x1, y1)
    if box == full_box:
        print(f"Auto crop: no black border detected; keeping {width}x{height}.")
    else:
        print(
            f"Auto crop: {width}x{height} -> {crop_width}x{crop_height}; "
            f"box=(x:{x0}-{x1}, y:{y0}-{y1}), samples={valid_samples}."
        )
    return box


def get_crop_box(video: str) -> tuple[int, int, int, int]:
    key = str(video)
    if key not in CROP_CACHE:
        CROP_CACHE[key] = detect_content_box(key)
    return CROP_CACHE[key]


def crop_frame(frame: np.ndarray, box: tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = box
    return frame[y0:y1, x0:x1]


def crop_around_pick(
    frame: np.ndarray,
    kp: np.ndarray,
    vis: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, bool]:
    """Return a fixed 640x480 crop centered on the visible pick keypoints."""
    frame_h, frame_w = frame.shape[:2]
    source_w = min(OUTPUT_WIDTH, frame_w)
    source_h = min(OUTPUT_HEIGHT, frame_h)

    valid = (
        np.asarray(vis, dtype=bool)
        & np.isfinite(kp).all(axis=1)
        & (kp[:, 0] >= 0)
        & (kp[:, 0] < frame_w)
        & (kp[:, 1] >= 0)
        & (kp[:, 1] < frame_h)
    )
    if valid.any():
        visible_kp = kp[valid]
        center_x = float(visible_kp[:, 0].min() + visible_kp[:, 0].max()) / 2
        center_y = float(visible_kp[:, 1].min() + visible_kp[:, 1].max()) / 2
    else:
        center_x, center_y = frame_w / 2, frame_h / 2

    x0 = int(np.clip(round(center_x - source_w / 2), 0, frame_w - source_w))
    y0 = int(np.clip(round(center_y - source_h / 2), 0, frame_h - source_h))
    x1, y1 = x0 + source_w, y0 + source_h
    cropped = frame[y0:y1, x0:x1]

    # Padding is only needed when the detected content is smaller than 640x480.
    pad_left = (OUTPUT_WIDTH - source_w) // 2
    pad_right = OUTPUT_WIDTH - source_w - pad_left
    pad_top = (OUTPUT_HEIGHT - source_h) // 2
    pad_bottom = OUTPUT_HEIGHT - source_h - pad_top
    if pad_left or pad_right or pad_top or pad_bottom:
        cropped = cv2.copyMakeBorder(
            cropped,
            pad_top,
            pad_bottom,
            pad_left,
            pad_right,
            cv2.BORDER_CONSTANT,
            value=(0, 0, 0),
        )

    kp_out = kp.copy()
    kp_out[:, 0] += pad_left - x0
    kp_out[:, 1] += pad_top - y0
    inside = (
        (kp_out[:, 0] >= 0)
        & (kp_out[:, 0] < OUTPUT_WIDTH)
        & (kp_out[:, 1] >= 0)
        & (kp_out[:, 1] < OUTPUT_HEIGHT)
    )
    vis_out = valid & inside
    contains_all_visible = bool(np.all(~valid | inside))

    assert cropped.shape[:2] == (OUTPUT_HEIGHT, OUTPUT_WIDTH)
    return cropped, kp_out, vis_out, contains_all_visible


def read_frame(video: str, idx: int) -> np.ndarray:
    cap = cv2.VideoCapture(video)
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"Could not read frame {idx}")
    return crop_frame(frame, get_crop_box(video))


CLICK_JS = '''
window.clickPoints = async function(b64, names) {
  const colors = ['#ff8c00','#ff00ff','#00e5ff','#00ff00','#ff3030'];
  const wrap = document.createElement('div');
  const msg = document.createElement('div');
  msg.style.cssText = 'font:16px sans-serif;margin:6px 0;padding:6px 10px;background:#222;color:#fff';
  const canvas = document.createElement('canvas');
  canvas.style.cssText = 'max-width:100%;cursor:crosshair;display:block';
  const bar = document.createElement('div'); bar.style.margin = '8px 0';
  const undo = document.createElement('button'); undo.textContent = 'Undo';
  const done = document.createElement('button'); done.textContent = 'Done'; done.style.marginLeft = '8px';
  bar.append(undo, done); wrap.append(msg, canvas, bar); document.body.appendChild(wrap);
  const img = new Image();
  await new Promise(r => { img.onload = r; img.src = 'data:image/jpeg;base64,' + b64; });
  canvas.width = img.width; canvas.height = img.height;
  const ctx = canvas.getContext('2d'); const pts = [];
  function draw() {
    ctx.drawImage(img, 0, 0);
    pts.forEach((p, i) => {
      ctx.fillStyle = colors[i]; ctx.beginPath(); ctx.arc(p[0], p[1], 7, 0, 2*Math.PI); ctx.fill();
      ctx.font = '20px sans-serif'; ctx.lineWidth = 4; ctx.strokeStyle = '#000';
      ctx.strokeText(names[i], p[0]+10, p[1]-10); ctx.fillText(names[i], p[0]+10, p[1]-10);
    });
    msg.textContent = pts.length < names.length
      ? 'Click ' + (pts.length + 1) + '/' + names.length + ': ' + names[pts.length]
      : 'All points placed - click Done (or Undo to fix one)';
    done.disabled = pts.length !== names.length;
  }
  canvas.onclick = e => {
    if (pts.length >= names.length) return;
    const r = canvas.getBoundingClientRect();
    pts.push([Math.round((e.clientX - r.left) * canvas.width / r.width),
              Math.round((e.clientY - r.top) * canvas.height / r.height)]);
    draw();
  };
  undo.onclick = () => { pts.pop(); draw(); };
  draw();
  await new Promise(r => done.onclick = r);
  bar.remove(); msg.textContent = 'Saved: ' + JSON.stringify(pts);
  return pts;
}
'''


def click_points(frame: np.ndarray, names: list[str]) -> list[list[int]]:
    ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
    if not ok:
        raise RuntimeError("Could not encode the click frame")
    b64 = base64.b64encode(buf).decode()
    display(Javascript(CLICK_JS))
    return eval_js(f"clickPoints('{b64}', {json.dumps(names)})")


def yolo_line(kp: np.ndarray, vis: np.ndarray, W: int, H: int, pad: float) -> str:
    kp = np.asarray(kp, dtype=float)
    vis = np.asarray(vis, dtype=bool)
    if kp.shape != (len(KEYPOINTS), 2) or vis.shape != (len(KEYPOINTS),):
        raise ValueError("Expected exactly five (x, y) keypoints and visibility flags")
    if W <= 0 or H <= 0 or pad < 0:
        raise ValueError("Invalid output dimensions or padding")
    valid = (vis & np.isfinite(kp).all(axis=1)
             & (kp[:, 0] >= 0) & (kp[:, 0] < W)
             & (kp[:, 1] >= 0) & (kp[:, 1] < H))
    if not valid.any():
        raise ValueError("Cannot create a YOLO label without a visible keypoint")
    xs = np.clip(kp[valid, 0], 0, W - 1)
    ys = np.clip(kp[valid, 1], 0, H - 1)
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    p = pad * max(x1 - x0, y1 - y0)
    x0, y0, x1, y1 = max(0, x0 - p), max(0, y0 - p), min(W - 1, x1 + p), min(H - 1, y1 + p)
    if x1 <= x0 or y1 <= y0:
        raise ValueError("Visible keypoints must define a nonzero-area box")
    parts = [str(CLASS_ID)] + [f"{v:.6f}" for v in ((x0+x1)/2/W, (y0+y1)/2/H, (x1-x0)/W, (y1-y0)/H)]
    for (x, y), v in zip(kp, valid):
        parts += [f"{x/W:.6f}", f"{y/H:.6f}", "2"] if v else ["0", "0", "0"]
    return " ".join(parts)


def write_data_yaml():
    os.makedirs(OUT, exist_ok=True)
    with open(os.path.join(OUT, "data.yaml"), "w") as f:
        f.write("# Annotation batch only: create disjoint splits before training.\n"
                "train: train/images\nval: val/images\n"
                f"kpt_shape: [{len(KEYPOINTS)}, 3]\nflip_idx: {list(range(len(KEYPOINTS)))}\n"
                f"names:\n  {CLASS_ID}: {CLASS_NAME}\n# keypoint order: {KEYPOINTS}\n")


def track_and_save(video, points, click_frame):
    if TRACK_STRIDE <= 0 or EVERY <= 0:
        raise ValueError("Sampling strides must be positive")
    clip = Path(video).stem
    for subdir in ("images", "labels"):
        if list((Path(OUT) / "train" / subdir).glob(f"{clip}_f*")):
            raise FileExistsError(f"Existing output for {clip}; use a new OUT folder for a rerun")
    assert EVERY % TRACK_STRIDE == 0, "EVERY must be a multiple of TRACK_STRIDE"
    assert click_frame % TRACK_STRIDE == 0, f"CLICK_FRAME must be a multiple of TRACK_STRIDE ({TRACK_STRIDE})"

    crop_box = get_crop_box(video)

    # Load every TRACK_STRIDE-th cropped frame, downscaled, for tracking.
    cap = cv2.VideoCapture(video)
    small = []
    W = H = None
    s = 1.0
    T = 0
    while True:
        ok, f = cap.read()
        if not ok:
            break
        f = crop_frame(f, crop_box)
        if W is None:
            H, W = f.shape[:2]
            s = min(1.0, MAX_SIDE / max(H, W))
        if T % TRACK_STRIDE == 0:
            if s < 1.0:
                f = cv2.resize(f, (round(W*s), round(H*s)), interpolation=cv2.INTER_AREA)
            small.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        T += 1
    cap.release()
    if not small or not (0 <= click_frame < T):
        raise ValueError("Video is empty or CLICK_FRAME is outside the video")
    if len(points) != len(KEYPOINTS) or not np.isfinite(points).all():
        raise ValueError("Click all five finite keypoints first")
    tracking_h, tracking_w = small[0].shape[:2]
    scale_xy = np.array([tracking_w / W, tracking_h / H])
    print(f"{T} frames cropped to {W}x{H}; tracking {len(small)} of them (every {TRACK_STRIDE})...")

    torch.cuda.empty_cache()
    vid = torch.from_numpy(np.stack(small)).permute(0, 3, 1, 2)[None].float().to(device)
    qf = click_frame // TRACK_STRIDE
    q = torch.tensor([[[qf, x*scale_xy[0], y*scale_xy[1]] for x, y in points]], dtype=torch.float32, device=device)
    with torch.no_grad():
        tracks, visible = model(vid, queries=q, backward_tracking=qf > 0)
    del vid, q
    torch.cuda.empty_cache()
    tracks = tracks[0].cpu().numpy() / scale_xy
    visible = visible[0].cpu().numpy() > 0.5
    visible &= np.isfinite(tracks).all(axis=-1)
    visible &= (tracks[..., 0] >= 0) & (tracks[..., 0] < W) & (tracks[..., 1] >= 0) & (tracks[..., 1] < H)

    clip = os.path.splitext(os.path.basename(video))[0]
    img_dir = os.path.join(OUT, "train", "images")
    lbl_dir = os.path.join(OUT, "train", "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    write_data_yaml()

    saved, skipped, clipped, previews = [], 0, 0, []
    cap = cv2.VideoCapture(video)
    for i in range(T):
        ok, frame = cap.read()
        if not ok:
            break
        frame = crop_frame(frame, crop_box)
        if i % EVERY:
            continue
        ti = i // TRACK_STRIDE
        if ti >= len(tracks):
            break
        kp, vis = tracks[ti], visible[ti]
        if vis.sum() < MIN_VISIBLE:
            skipped += 1
            continue
        frame, kp, vis, contains_all = crop_around_pick(frame, kp, vis)
        if vis.sum() < MIN_VISIBLE:
            skipped += 1
            continue
        if not contains_all:
            clipped += 1
        name = f"{clip}_f{i:05d}"
        image_path = os.path.join(img_dir, name + ".jpg")
        if not cv2.imwrite(image_path, frame):
            raise OSError(f"Could not write image: {image_path}")
        with open(os.path.join(lbl_dir, name + ".txt"), "w") as f:
            f.write(yolo_line(kp, vis, OUTPUT_WIDTH, OUTPUT_HEIGHT, PAD) + "\n")
        saved.append(i)
        disp = frame.copy()
        for j, ((x, y), v) in enumerate(zip(kp, vis)):
            if v and np.isfinite([x, y]).all():
                cv2.circle(disp, (int(x), int(y)), 8, COLORS_BGR[j], -1)
        previews.append((i, cv2.resize(disp, (OUTPUT_WIDTH // 2, OUTPUT_HEIGHT // 2))))
    cap.release()
    print(
        f"Saved {len(saved)} images at {OUTPUT_WIDTH}x{OUTPUT_HEIGHT} from {clip}; "
        f"skipped {skipped} (too few visible points), "
        f"{clipped} crops could not contain every visible point."
    )
    return clip, previews


def show_previews(clip: str, previews: list, cols: int = 4, max_frames: int = 24) -> None:
    # Bound rendered output; inspect additional ranges if a clip shows tracking drift.
    if len(previews) > max_frames:
        indices = np.linspace(0, len(previews) - 1, max_frames, dtype=int)
        previews = [previews[index] for index in indices]
    if not previews:
        print(f"{clip}: no preview frames were saved.")
        return
    rows = math.ceil(len(previews) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 2.6 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (i, img) in zip(axes.flat, previews):
        ax.imshow(img[..., ::-1])
        ax.set_title(f"frame {i}", fontsize=10)
    plt.suptitle(f"{clip}: visible points only; review every exported frame before training", y=1.0)
    plt.tight_layout()
    plt.show()


print(
    f"Ready. Black borders are removed first, then the pick is cropped to "
    f"{OUTPUT_WIDTH}x{OUTPUT_HEIGHT}; output goes to {OUT}."
)


In [ ]:
# 3. Select one clip from Google Drive
# Choose a video from the dropdown, then click "Load selected video".
import re
import ipywidgets as widgets
from IPython.display import clear_output

def natural_key(path):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", path.name)
    ]

drive_videos = sorted(
    [path for path in VIDEO_DIR.iterdir()
     if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS],
    key=natural_key,
)
assert drive_videos, f"No videos found in: {VIDEO_DIR}"
video_by_name = {video.name: video for video in drive_videos}

video_dropdown = widgets.Dropdown(
    options=list(video_by_name),
    description="Video:",
    layout=widgets.Layout(width="500px"),
)
load_button = widgets.Button(
    description="Load selected video",
    button_style="primary",
)
load_output = widgets.Output()

def load_selected_video(_):
    global VIDEO, SOURCE_VIDEO
    with load_output:
        clear_output()

        for variable in ("POINTS", "PREVIEWS", "CLIP", "REVIEWED_CLIP", "VIDEO", "SOURCE_VIDEO"):
            globals().pop(variable, None)
        gc.collect()
        torch.cuda.empty_cache()

        source_video = video_by_name[video_dropdown.value]
        local_dir = Path("/content/clips")
        local_dir.mkdir(parents=True, exist_ok=True)
        local_video = local_dir / source_video.name
        shutil.copy2(source_video, local_video)

        cap = cv2.VideoCapture(str(local_video))
        if not cap.isOpened():
            raise RuntimeError(f"Could not open video: {local_video}")
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()

        if n <= 0 or w <= 0 or h <= 0:
            raise RuntimeError(f"Invalid video metadata: {local_video}")
        CROP_CACHE.pop(str(local_video), None)
        SOURCE_VIDEO = source_video
        VIDEO = str(local_video)
        print(f"Loaded from Drive: {source_video.name}")
        print(f"{n} frames, {fps:.1f} fps, {n / max(fps, 1):.1f} s, {w}x{h}")
        print("Ready. Continue with Part 4.")

load_button.on_click(load_selected_video)
display(video_dropdown, load_button, load_output)
print(f"Found {len(drive_videos)} videos. Select one and click the button.")


In [ ]:
# 4. Pick a clear frame where ALL five keypoints are visible, then click in order.
# Use a multiple of TRACK_STRIDE: e.g. 0, 8, 16, 80 for the default stride.
assert "VIDEO" in globals(), "Select a video and click Load selected video in Cell 3 first."
CLICK_FRAME = 0
assert CLICK_FRAME >= 0 and CLICK_FRAME % TRACK_STRIDE == 0
POINTS = click_points(read_frame(VIDEO, CLICK_FRAME), KEYPOINTS)
print(dict(zip(KEYPOINTS, POINTS)))


In [ ]:
# 5. Track and export local image/label pairs, then inspect the preview.
# CUDA OOM: restart, shorten the clip or increase TRACK_STRIDE (e.g. 16).
# Keep EVERY a multiple of TRACK_STRIDE and CLICK_FRAME divisible by it.
# Use a new OUT folder when rerunning a clip to avoid mixing old and new labels.
assert "POINTS" in globals(), "Click the five points in Cell 4 first."
globals().pop("REVIEWED_CLIP", None)
CLIP, PREVIEWS = track_and_save(VIDEO, POINTS, CLICK_FRAME)
show_previews(CLIP, PREVIEWS)


In [ ]:
# 6. Reject frames with fewer than five visible keypoints and manually flagged frames.
# Inspect previews and add drifting/incorrect frame numbers to BAD, then run this cell.
# Rejected pairs are MOVED to OUT/rejected/ (local, not included in the ZIP).
assert "CLIP" in globals(), "Run Cell 5 first."
BAD = []  # e.g. [96, 104]

image_dir = Path(OUT) / "train" / "images"
label_dir = Path(OUT) / "train" / "labels"
expected_tokens = 5 + 3 * len(KEYPOINTS)
rejected = []

# Validate every label before moving any files.
for label_path in sorted(label_dir.glob(f"{CLIP}_f*.txt")):
    tokens = label_path.read_text().split()
    if len(tokens) != expected_tokens:
        raise ValueError(f"Malformed label: {label_path.name}")
    values = np.asarray(tokens, dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f"Nonfinite label: {label_path.name}")
    visibility = values[7::3]
    if not np.isin(visibility, [0, 1, 2]).all():
        raise ValueError(f"Invalid visibility: {label_path.name}")
    frame_number = int(label_path.stem.rsplit("_f", 1)[1])
    if (visibility > 0).sum() < len(KEYPOINTS) or frame_number in BAD:
        image_path = image_dir / f"{label_path.stem}.jpg"
        if not image_path.exists():
            raise FileNotFoundError(image_path)
        for path in (image_path, label_path):
            destination = Path(OUT) / "rejected" / path.parent.name / path.name
            if destination.exists():
                raise FileExistsError(destination)
        rejected.append((image_path, label_path))

for pair in rejected:
    for path in pair:
        destination = Path(OUT) / "rejected" / path.parent.name / path.name
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(path), str(destination))

REVIEWED_CLIP = CLIP
print(f"Moved {len(rejected)} rejected pairs into {Path(OUT) / 'rejected'}")
print(f"Remaining images for {CLIP}: {len(list(image_dir.glob(f'{CLIP}_f*.jpg')))}")
print("Run Cell 7 to export this clip, then return to Cell 3 for the next clip.")


In [ ]:
# 7. Save ONLY the current clip as a ZIP on Drive; existing ZIPs are preserved.
import tempfile
from pathlib import Path

out_path = Path(OUT)

assert "CLIP" in globals(), "Please run Cell 5 first."
assert globals().get("REVIEWED_CLIP") == CLIP, "Run Cell 6 to review this clip first."
assert out_path.exists(), f"Output directory not found: {out_path}"
assert ANNOTATED_DIR.exists(), f"Annotated directory not found: {ANNOTATED_DIR}"

image_dir = out_path / "train" / "images"
label_dir = out_path / "train" / "labels"

# Only select files belonging to the current video
images = sorted(image_dir.glob(f"{CLIP}_f*.jpg"))
labels = sorted(label_dir.glob(f"{CLIP}_f*.txt"))

assert images, f"No labeled images found for {CLIP}."
assert len(images) == len(labels), (
    f"Image/label count mismatch for {CLIP}: "
    f"{len(images)} images, {len(labels)} labels"
)

image_stems = {path.stem for path in images}
label_stems = {path.stem for path in labels}
assert image_stems == label_stems, "Some images do not have matching labels."

zip_base = ANNOTATED_DIR / f"rock_pick_labels_{CLIP}"
zip_path = zip_base.parent / (zip_base.name + ".zip")

# Prevent accidental overwrite
assert not zip_path.exists(), (
    f"{zip_path.name} already exists. "
    "Rename or delete the existing ZIP before running this cell again."
)

# Build a temporary YOLO dataset containing only this video
with tempfile.TemporaryDirectory() as temp_dir:
    package_dir = Path(temp_dir)
    package_images = package_dir / "train" / "images"
    package_labels = package_dir / "train" / "labels"

    package_images.mkdir(parents=True)
    package_labels.mkdir(parents=True)

    for image in images:
        shutil.copy2(image, package_images / image.name)

    for label in labels:
        shutil.copy2(label, package_labels / label.name)

    shutil.copy2(out_path / "data.yaml", package_dir / "data.yaml")

    ZIP_PATH = shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=str(package_dir),
    )

print(f"Saved ZIP to Drive: {ZIP_PATH}")
print(f"Video: {CLIP}")
print(f"Included {len(images)} images and {len(labels)} labels.")
